In [ ]:
from pathlib import Path
from io import BytesIO
import json, csv, hashlib, math

PACKAGE_ROOT = next((p for p in (Path.cwd(), Path.cwd().parent)
                     if (p / "data").is_dir() and (p / "notebooks").is_dir()), None)
if PACKAGE_ROOT is None:
    raise FileNotFoundError("Open this notebook from the package root or notebooks directory.")

SAVE_PDF = False

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.stats import t
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D
from IPython.display import display, Image
plt.rcdefaults()

def load_npz(relative_path):
    with np.load(PACKAGE_ROOT / relative_path, allow_pickle=False) as archive:
        return {key: archive[key].copy() for key in archive.files}

def add_panel_labels(figure):
    panels = [axis for axis in figure.axes if axis.get_label() != "<colorbar>"]
    assert len(panels) == 8, "Unexpected panel count"
    for index, axis in enumerate(panels):
        axis.text(-0.16, 1.04, f"({chr(97 + index)})",
                  transform=axis.transAxes, fontsize=12, fontweight="bold",
                  ha="left", va="bottom", clip_on=False, gid="panel_label")
    figure.canvas.draw()
    renderer = figure.canvas.get_renderer()
    records = []
    for axis in panels:
        label = next(text for text in axis.texts if text.get_gid() == "panel_label")
        box = label.get_window_extent(renderer)
        for title in (axis.title, axis._left_title, axis._right_title):
            if title.get_text():
                assert not box.overlaps(title.get_window_extent(renderer)), "Panel label overlaps title"
        records.append({"label": label.get_text(), "axes_position": list(axis.get_position().bounds),
                        "label_bbox_pixels": list(box.bounds)})

def export_figure(figure, figure_id, tight=False):
    add_panel_labels(figure)
    options = {"bbox_inches": "tight"} if tight else {}
    preview = BytesIO()
    figure.savefig(preview, format="png", dpi=180, **options)
    display(Image(data=preview.getvalue()))
    if SAVE_PDF:
        output = PACKAGE_ROOT / "outputs"
        output.mkdir(exist_ok=True)
        figure.savefig(output / f"{figure_id}.pdf",
                       metadata={"CreationDate": None, "ModDate": None}, **options)
        print(f"Saved outputs/{figure_id}.pdf")


In [ ]:
from matplotlib.colors import to_rgba
ROOT=PACKAGE_ROOT/"data/Fig5"
SCOPES = [
    (r'$N=25$', 25, '#CC79A7', 'D'),
    (r'$N=50$', 50, '#E69F00', 'v'),
    (r'$N=100$', 100, '#009E73', '^'),
    (r'$N=200$', 200, '#0072B2', 'o'),
    (r'$N=500$', 500, '#D55E00', 's'),
]
TITLES = {
    'noiseless': 'Noiseless',
    'pauli_y_003': 'Encoding Pauli-Z\n' + r'$p_z=0.03$',
    'pauli_y_006': 'Encoding Pauli-Z\n' + r'$p_z=0.06$',
    'symmetric_0025': 'Encoding symmetric Pauli\n' + r'$p=0.025$',
    'symmetric_0005': 'Encoding symmetric Pauli\n' + r'$p=0.005$',
    'allblock_encoder_0005': 'Encoding + block Pauli\n' + r'$p_{\rm enc}=0.005$',
    'thermal_weak': 'CZ-only thermal relaxation\n' + r'$\tau=0.005$, $T_2=T_1$',
    'thermal_matched': 'CZ-only thermal relaxation\n' + r'$\tau=0.0100503$, $T_2=T_1$',
}
TITLE_REFERENCE = json.loads((ROOT / 'provenance/maintext_fig4_titles.json').read_text())
assert TITLES == TITLE_REFERENCE['titles']
STEM = 'GD20_cutoff_loss_change_thermal8_early25_50_ylabel_corrected'


def digest(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()




def compute(manifest):
    cutoffs = np.asarray(manifest['cutoffs'])
    window = manifest['window']
    assert window == 10 and manifest['endpoints'] == [25, 50, 100, 200, 500]
    summary, per_seed_rows, window_rows, checks = [], [], [], {}
    products = {}
    for condition in manifest['conditions']:
        key = condition['key']
        path = ROOT / condition['file']
        assert digest(path) == condition['sha256'], path
        with np.load(path, allow_pickle=False) as z:
            modes = z['mode_loss_0_39'].copy()
            total = z['total_loss'].copy()
            tail = z['tail_loss_gt39'].copy()
            seeds = z['seeds'].copy()
            np.testing.assert_array_equal(seeds, np.arange(42, 62))
            np.testing.assert_array_equal(z['steps'], np.arange(501))
            assert modes.shape == (20, 501, 40) and total.shape == (20, 501)
            assert np.isfinite(modes).all() and np.isfinite(total).all()
            assert np.all(modes >= 0) and np.all(total > 0)
            np.testing.assert_allclose(modes.sum(-1) + tail, total, rtol=1e-12, atol=1e-14)
            cumulative = np.stack([modes[:, :, :k + 1].sum(-1) for k in cutoffs], axis=-1)
            delta_total = total[:, window:] - total[:, :-window]
            delta_cumulative = cumulative[:, window:] - cumulative[:, :-window]
            threshold = manifest['denominator_relative_threshold'] * total[:, :1]
            valid = np.abs(delta_total) > threshold
            ratio = np.divide(delta_cumulative, delta_total[..., None],
                              out=np.full_like(delta_cumulative, np.nan),
                              where=valid[..., None])
            tail_at_cutoff = total[..., None] - cumulative
            alternative = 1 - np.divide(tail_at_cutoff[:, window:] - tail_at_cutoff[:, :-window],
                                        delta_total[..., None],
                                        out=np.full_like(delta_cumulative, np.nan),
                                        where=valid[..., None])
            np.testing.assert_allclose(ratio, alternative, rtol=1e-8, atol=1e-8, equal_nan=True)
            sidecar_difference = None
            if 'reference_derivative_ratio_w10' in z.files:
                np.testing.assert_array_equal(z['reference_cutoffs'], cutoffs)
                np.testing.assert_array_equal(z['reference_valid_mask_w10'], valid)
                np.testing.assert_allclose(ratio, z['reference_derivative_ratio_w10'],
                                           rtol=2e-8, atol=1e-8, equal_nan=True)
                sidecar_difference = float(np.nanmax(np.abs(ratio-z['reference_derivative_ratio_w10'])))
        products[key] = {}
        for label, endpoint, color, marker in SCOPES:
            count = endpoint - window + 1
            selected = ratio[:, :count]
            mask = valid[:, :count]
            assert np.all(mask.any(axis=1)), (key, endpoint)
            per_seed = np.stack([selected[s, mask[s]].mean(axis=0) for s in range(20)])
            mean = per_seed.mean(axis=0)
            half = t.ppf(.975, 19) * per_seed.std(axis=0, ddof=1) / np.sqrt(20)
            products[key][endpoint] = (mean, half)
            for i, k in enumerate(cutoffs):
                summary.append({'setting': key, 'max_iteration': endpoint, 'window_steps': window,
                                'cutoff_k': int(k), 'lambda': int(k*k), 'ratio_mean': float(mean[i]),
                                'ci95_halfwidth': float(half[i]), 'ci95_lower': float(mean[i]-half[i]),
                                'ci95_upper': float(mean[i]+half[i]), 'seed_count': 20,
                                'valid_window_fraction': float(mask.mean())})
                for s, seed in enumerate(seeds):
                    per_seed_rows.append({'setting': key, 'seed': int(seed), 'max_iteration': endpoint,
                                          'cutoff_k': int(k), 'ratio_mean_over_valid_windows': float(per_seed[s, i])})
            for s, seed in enumerate(seeds):
                window_rows.append({'setting': key, 'seed': int(seed), 'max_iteration': endpoint,
                                    'candidate_windows': count, 'valid_windows': int(mask[s].sum()),
                                    'positive_total_change_windows': int(np.sum(delta_total[s, :count] > 0)),
                                    'denominator_threshold': float(threshold[s, 0])})
        checks[key] = {'valid_window_fraction': float(valid.mean()),
                       'total_loss_closure_max_abs': float(np.max(np.abs(total - modes.sum(-1) - tail))),
                       'complement_ratio_max_abs_difference': float(np.nanmax(np.abs(ratio-alternative))),
                       'reference_sidecar_max_abs_difference': sidecar_difference,
                       'all_20_seeds_in_all_scopes': True}
    assert list(products) == list(TITLES)
    return products, summary, per_seed_rows, window_rows, checks


def bounds_check(fig):
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    artists = [*fig.legends, *fig.texts]
    for ax in fig.axes:
        artists.extend([ax.title, ax.xaxis.label, ax.yaxis.label])
        artists.extend(ax.get_xticklabels())
        artists.extend([label for tick, label in zip(ax.get_yticks(), ax.get_yticklabels())
                        if ax.get_ylim()[0] <= tick <= ax.get_ylim()[1]])
    for artist in artists:
        if not artist.get_visible():
            continue
        box = artist.get_window_extent(renderer)
        assert box.x0 >= -1 and box.y0 >= -1 and box.x1 <= fig.bbox.x1+1 and box.y1 <= fig.bbox.y1+1, str(artist)


def plot(products, manifest):
    plt.rcParams.update({'font.family': 'serif', 'font.serif': ['DejaVu Serif'],
                         'font.size': 10, 'axes.titlesize': 10.5, 'axes.labelsize': 11,
                         'xtick.labelsize': 9, 'ytick.labelsize': 9, 'legend.fontsize': 10,
                         'axes.linewidth': .8, 'pdf.fonttype': 42, 'ps.fonttype': 42})
    fig, axes = plt.subplots(2, 4, figsize=(14.0, 7.5), sharex=True, sharey=True)
    cutoffs = np.asarray(manifest['cutoffs'])
    lo = min(float(np.min(mean-half)) for scopes in products.values() for mean, half in scopes.values())
    hi = max(1., max(float(np.max(mean+half)) for scopes in products.values() for mean, half in scopes.values()))
    margin = .07 * (hi - min(lo, 0.))
    limits = (min(lo, 0.) - margin, hi + margin)
    for ax, (key, scopes) in zip(axes.flat, products.items()):
        for label, endpoint, color, marker in SCOPES:
            mean, half = scopes[endpoint]
            line, = ax.plot(cutoffs, mean, color=color, marker=marker, markersize=4,
                            linewidth=1.25, label=label)
            np.testing.assert_array_equal(line.get_ydata(), mean)
            np.testing.assert_allclose(to_rgba(line.get_color()), to_rgba(color))
            ax.fill_between(cutoffs, mean-half, mean+half, color=color, alpha=.13, linewidth=0)
            assert np.min(mean-half) > limits[0] and np.max(mean+half) < limits[1]
        ax.axhline(1, color='.35', linestyle='--', linewidth=.8, zorder=1)
        title_scale = fig.get_figwidth() / TITLE_REFERENCE['reference_figure_width_inches']
        ax.set_title(TITLES[key],
                     fontsize=TITLE_REFERENCE['title_fontsize_points'] * title_scale,
                     pad=TITLE_REFERENCE['title_pad_points'] * title_scale)
        ax.set_xticks(cutoffs)
        ax.set_xlim(-.5, 40.5)
        ax.set_ylim(*limits)
        ax.grid(alpha=.16)
        ax.tick_params(direction='out', length=3, width=.7)
    for ax in axes[1]:
        ax.set_xlabel(r'Frequency cutoff $K$')
    for ax in axes[:, 0]:
        ax.set_ylabel(r'Mean loss-change ratio $R_K(N)$')
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(.52, .995),
               ncol=5, frameon=False, handlelength=2.3, columnspacing=2.0)
    fig.subplots_adjust(left=.066, right=.985, bottom=.10, top=.86, hspace=.41, wspace=.16)
    bounds_check(fig)
    return fig, limits

In [ ]:
manifest=json.loads((ROOT/"manifest.json").read_text())
products,summary,seeds,windows,checks=compute(manifest)
fig,limits=plot(products,manifest)
export_figure(fig,"Fig5")
plt.close(fig)